# Day 075 — Exercise 4: add_captions

**What you'll build:** `add_captions(video_path, text, output_path, fontsize, color, ffmpeg_fn=None) -> Path` — burn caption text onto a video using FFmpeg's `drawtext` filter.

**Why it matters:** Stage 4 of the pipeline — captions make the video accessible and allow the lesson text to be read even without audio.

In [ ]:
from pathlib import Path
import tempfile, subprocess
_mock_caption_fn = lambda video_path, text, output_path: (Path(output_path).write_bytes(b'CAP' + bytes(len(text))), Path(output_path))[1]


## Task

- **Mock:** `return ffmpeg_fn(video_path, text, output_path)`
- **Real:** escape `text` (`'` → `\'`, `:` → `\:`); run `ffmpeg -y -i video_path -vf drawtext=text='{safe_text}':fontsize={fontsize}:fontcolor={color}:x=(w-text_w)/2:y=h-text_h-20 output_path`
- Raise `RuntimeError` if `returncode != 0`; return `Path(output_path)`

## Your Implementation

In [ ]:
def add_captions(video_path, text: str, output_path,
                 fontsize: int = 24, color: str = 'white',
                 ffmpeg_fn=None):
    """Burn a caption onto a video using FFmpeg drawtext filter.

    Args:
        video_path:  source video
        text:        caption text (single line)
        output_path: destination path
        fontsize:    font size in pixels
        color:       text colour name ('white', 'yellow', 'black', ...)
        ffmpeg_fn:   callable(video_path, text, output_path) -> Path
    Returns:
        Path to captioned video
    """
    raise NotImplementedError


In [ ]:
def add_captions(video_path, text, output_path,
                 fontsize=24, color='white', ffmpeg_fn=None):
    if ffmpeg_fn is not None:
        return ffmpeg_fn(video_path, text, output_path)
    safe_text = text.replace("'", r"\'").replace(':', r'\:')
    out_path = Path(output_path)
    result = subprocess.run(
        ['ffmpeg', '-y', '-i', str(video_path),
         '-vf', (f"drawtext=text='{safe_text}':fontsize={fontsize}:"
                 f"fontcolor={color}:x=(w-text_w)/2:y=h-text_h-20"),
         str(out_path)],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(f'FFmpeg caption error: {result.stderr[-500:]}')
    return out_path


## Automated checks

In [ ]:

score, total = 0, 5
try:
    with tempfile.NamedTemporaryFile(suffix='.mp4', delete=False) as f:
        src = f.name
    Path(src).write_bytes(b'VIDEO')

    with tempfile.NamedTemporaryFile(suffix='.mp4', delete=False) as f:
        out1 = f.name
    with tempfile.NamedTemporaryFile(suffix='.mp4', delete=False) as f:
        out2 = f.name

    # returns Path
    result = add_captions(src, 'Hello Day 75', out1, ffmpeg_fn=_mock_caption_fn)
    assert isinstance(result, Path)
    score += 1; print("✅ returns Path")

    # file exists and non-empty
    assert result.exists() and result.stat().st_size > 0
    score += 1; print("✅ output file exists with non-zero size")

    # ffmpeg_fn receives (video_path, text, output_path)
    captured = {}
    def _cap(vp, t, op):
        captured.update(text=t, op=op)
        return Path(op)
    add_captions(src, 'Day 75', out1, ffmpeg_fn=_cap)
    assert captured.get('text') == 'Day 75'
    score += 1; print("✅ text forwarded to ffmpeg_fn correctly")

    # output_path is respected
    assert str(result) == out1
    score += 1; print("✅ returned Path matches output_path")

    # different text lengths → different output sizes (text encoded in mock)
    r1 = add_captions(src, 'Hi', out1, ffmpeg_fn=_mock_caption_fn)
    r2 = add_captions(src, 'Hello World Pipeline', out2, ffmpeg_fn=_mock_caption_fn)
    assert r1.stat().st_size != r2.stat().st_size
    score += 1; print("✅ text incorporated (different lengths → different sizes)")

except Exception as e:
    print(f"❌ {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
def add_captions(video_path, text, output_path,
                 fontsize=24, color='white', ffmpeg_fn=None):
    if ffmpeg_fn is not None:
        return ffmpeg_fn(video_path, text, output_path)
    safe_text = text.replace("'", r"\'").replace(':', r'\:')
    out_path = Path(output_path)
    result = subprocess.run(
        ['ffmpeg', '-y', '-i', str(video_path),
         '-vf', (f"drawtext=text='{safe_text}':fontsize={fontsize}:"
                 f"fontcolor={color}:x=(w-text_w)/2:y=h-text_h-20"),
         str(out_path)],
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        raise RuntimeError(f'FFmpeg caption error: {result.stderr[-500:]}')
    return out_path
```

**drawtext positioning:** `x=(w-text_w)/2` centres horizontally (video width minus text width, halved). `y=h-text_h-20` places the text 20 pixels above the bottom edge. These expressions are evaluated at render time by FFmpeg against the actual frame dimensions.

</details>